# Modeling — IEEE-CIS Fraud Detection

This notebook loads the **processed** features saved by `ieee_cis_fraud_detection/features.py`
(`data/processed/*.parquet`), sets up a **temporal train/validation split** (the data is
time-ordered by `TransactionDT`), and provides an **MLflow** harness for tracking experiments.

You build the models — the cells below give you:
1. Loaded features + temporal split (`X_train`, `y_train`, `X_val`, `y_val`)
2. An MLflow setup + a `train_and_eval()` helper that logs validation AUC
3. Template examples to verify the harness works end-to-end

In [1]:
import numpy as np
import pandas as pd

import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ieee_cis_fraud_detection.config import PROCESSED_DATA_DIR, PROJ_ROOT

/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-26 17:25:44.712 | INFO     | ieee_cis_fraud_detection.config:<module>:11 - PROJ_ROOT path is: /Users/alex/IEEE-CIS_Fraud_Detection_MLOp


In [2]:
transaction = pd.read_parquet(PROCESSED_DATA_DIR / "train_transaction_filtered.parquet")
identity = pd.read_parquet(PROCESSED_DATA_DIR / "train_identity_filtered.parquet")

print("train_transaction:", transaction.shape)
print("train_identity:  ", identity.shape)
print(
    "Categorical cols (transaction):",
    [c for c in transaction.columns if transaction[c].dtype.name == "category"],
)

train_transaction: (590540, 220)
train_identity:   (144233, 29)
Categorical cols (transaction): ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M6']


## Baseline scope: transaction-only

We start with a **transaction-only** baseline and
left-join `identity` later if performance plateaus. The `identity` table is
loaded above and kept ready for that step.

In [3]:
TARGET = "isFraud"
# Pure ID column — never a feature. TransactionDT is kept as a feature for now
# (you may drop it or engineer hour/weekday features from it).
DROP_COLS = ["TransactionID"]


def prepare_data(df: pd.DataFrame):
    """Split a transaction frame into features and labels."""
    y = df[TARGET].astype(int).to_numpy()
    X = df.drop(columns=[TARGET] + DROP_COLS)
    return X, y


def temporal_split(df: pd.DataFrame, val_frac: float = 0.2):
    """Split by time (TransactionDT), NOT randomly — avoids time leakage."""
    df = df.sort_values("TransactionDT").reset_index(drop=True)
    split_idx = int(len(df) * (1 - val_frac))
    return df.iloc[:split_idx], df.iloc[split_idx:]


train_df, val_df = temporal_split(transaction, val_frac=0.2)
X_train, y_train = prepare_data(train_df)
X_val, y_val = prepare_data(val_df)

print(f"train: {X_train.shape}  val: {X_val.shape}")
print(f"val fraud rate: {y_val.mean():.4f}")

train: (472432, 218)  val: (118108, 218)
val fraud rate: 0.0344


## Why a temporal split?

`TransactionDT` is a timedelta from a fixed reference datetime, so rows are
**time-ordered**. A random KFold would place the same card/device in both train
and validation, inflating AUC with leakage. Sorting by `TransactionDT` and
holding out the last 20% of time gives an honest estimate of future fraud.

> Once defined here, this split is shared by every model family so all AUCs are
> directly comparable.

In [5]:
# --- MLflow setup ---------------------------------------------------------
# MLflow 3.x requires a database backend; sqlite keeps it fully local.
TRACKING_DB = PROJ_ROOT / "mlruns" / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB}")
mlflow.set_experiment("ieee-fraud-detection")
mlflow.autolog()  # auto-logs params/metrics for sklearn / lightgbm / xgboost / catboost


2026/08/26 17:26:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/26 17:26:15 INFO mlflow.store.db.utils: Updating database tables
2026/08/26 17:26:16 INFO mlflow.tracking.fluent: Experiment with name 'ieee-fraud-detection' does not exist. Creating a new experiment.
2026/08/26 17:26:23 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


In [11]:
def train_and_eval(model, X_tr, y_tr, X_va, y_va, run_name):
    """Fit a model inside an MLflow run and log the validation AUC."""
    with mlflow.start_run(run_name=run_name):
        model.fit(X_tr, y_tr)
        y_prob = model.predict_proba(X_va)[:, 1]
        auc = roc_auc_score(y_va, y_proba=y_prob)  # y_proba kwarg for sklearn >= 1.9
        mlflow.log_metric("val_auc", auc)
        print(f"{run_name}: val AUC = {auc:.4f}")
        return model, auc


## Build your own models

Call `train_and_eval(model, X_train, y_train, X_val, y_val, run_name="...")`
for each model you want to compare. Every run is logged to MLflow with its
`val_auc`.

**Two tracks (per our plan):**

- **Tree track (NaN-native)** — LightGBM / XGBoost / CatBoost can be fed
  `X_train` directly; they handle NaN and the `category` dtype natively.
- **Classical track** — LR / LDA / SVM / MLP need the `preprocessor` below
  (impute → one-hot categoricals → scale numerics).

The next cells are templates to verify the harness; replace/extend them freely.

In [7]:
# Shared preprocessing for the classical track (impute -> encode -> scale).
# IMPORTANT: fits only on X_train, so no validation information leaks.
categorical_features = X_train.select_dtypes(include="category").columns.tolist()
numeric_features = [c for c in X_train.columns if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)
print(
    f"numeric features: {len(numeric_features)} | "
    f"categorical features: {len(categorical_features)}"
)

numeric features: 209 | categorical features: 9


In [8]:
# EXAMPLE (classical track) — verify the harness, then tune/replace.
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])
model_lr, auc_lr = train_and_eval(lr_pipeline, X_train, y_train, X_val, y_val, "example_logreg")

2026/08/26 17:27:02 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/26 17:28:00 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_

example_logreg: val AUC = 0.8150


In [9]:
# EXAMPLE (tree track) — NaN-native, uses the `category` dtype directly.
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05)
model_lgb, auc_lgb = train_and_eval(lgb_model, X_train, y_train, X_val, y_val, "example_lgbm")

2026/08/26 17:28:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2026/08/26 17:29:20 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.129478 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16051
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 218
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784


2026/08/26 17:29:28 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/26 17:29:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/26 17:29:

example_lgbm: val AUC = 0.9006


## Compare runs

Run metadata is stored in `mlruns/mlflow.db` (SQLite) and model artifacts under
`mlruns/`. To open the UI, run `mlflow ui` from the project root and open the
printed URL.


In [10]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
exp = client.get_experiment_by_name("ieee-fraud-detection")
runs = client.search_runs([exp.experiment_id], order_by=["metrics.val_auc DESC"])

print(f"{'run':<24} {'val_auc':>8}")
print("-" * 34)
for r in runs:
    name = r.data.tags.get("mlflow.runName", "?")
    auc = r.data.metrics.get("val_auc", float("nan"))
    print(f"{name:<24} {auc:>8.4f}")

run                       val_auc
----------------------------------
example_lgbm               0.9006
example_logreg             0.8150
